# Signal Agent: MCap + EOD Backfill

This notebook implements Steps 3-5 from the architecture redesign:
1. **Step 3:** Fetch MCap via `/metrics/refresh`, then UPDATE `daily_calls` MCap fields
2. **Step 4:** Fetch EOD prices from CryptoCompare histoday for Apr 8-13
3. **Step 5:** Compute efficiency metrics and batch UPDATE `daily_calls`

Database is single-threaded; safe to iterate without locks.

In [2]:
import os
import sqlite3
from datetime import datetime, timezone, timedelta
from zoneinfo import ZoneInfo
import httpx
import json

# Database path
DB_PATH = os.environ.get("DB_PATH", "data/signals.db")
print(f"Using database: {DB_PATH}")

# ET timezone
ET = ZoneInfo("America/New_York")

def get_conn():
    conn = sqlite3.connect(DB_PATH, detect_types=sqlite3.PARSE_DECLTYPES)
    conn.row_factory = sqlite3.Row
    return conn

# Test connection
conn = get_conn()
cursor = conn.execute("SELECT COUNT(*) as cnt FROM daily_calls")
total_rows = cursor.fetchone()["cnt"]
print(f"Total rows in daily_calls: {total_rows}")
conn.close()

Using database: data/signals.db


OperationalError: unable to open database file

## Step 3: Update MCap fields in daily_calls from metrics_cache

After calling `POST /metrics/refresh` from frontend, metrics_cache is populated.
This cell UPDATEs all rows in daily_calls where `dq_mcap_missing = 1`.

In [ ]:
def update_daily_calls_mcap():
    """
    UPDATE daily_calls with MCap + tier from metrics_cache.
    Joins on ticker, fills mcap_at_call, mcap_tier, clears dq_mcap_missing flag.
    """
    conn = get_conn()
    try:
        # Count before
        before = conn.execute(
            "SELECT COUNT(*) as cnt FROM daily_calls WHERE dq_mcap_missing = 1"
        ).fetchone()["cnt"]
        print(f"Rows with dq_mcap_missing=1 before: {before}")

        # UPDATE: join metrics_cache and fill MCap fields
        update_sql = """
        UPDATE daily_calls
        SET
            mcap_at_call = (SELECT mcap FROM metrics_cache WHERE metrics_cache.ticker = daily_calls.ticker),
            mcap_tier = (SELECT mcap_tier FROM metrics_cache WHERE metrics_cache.ticker = daily_calls.ticker),
            dq_mcap_missing = 0
        WHERE dq_mcap_missing = 1
        AND ticker IN (SELECT DISTINCT ticker FROM metrics_cache)
        """
        conn.execute(update_sql)
        conn.commit()

        # Count after
        after = conn.execute(
            "SELECT COUNT(*) as cnt FROM daily_calls WHERE dq_mcap_missing = 1"
        ).fetchone()["cnt"]
        print(f"Rows with dq_mcap_missing=1 after: {after}")
        print(f"✅ Updated {before - after} rows")

        # Show sample
        sample = conn.execute(
            "SELECT ticker, et_day, activity_type, mcap_at_call, mcap_tier FROM daily_calls WHERE mcap_at_call IS NOT NULL LIMIT 5"
        ).fetchall()
        print("\nSample rows:")
        for row in sample:
            print(f"  {row['ticker']:8} {row['et_day']} {row['activity_type']:4} MCap={row['mcap_at_call']:>15,.0f} Tier={row['mcap_tier']}")

    finally:
        conn.close()

# Call after /metrics/refresh has been hit from frontend
# update_daily_calls_mcap()
print("Ready to call update_daily_calls_mcap() after /metrics/refresh populates metrics_cache")

## Step 4: Fetch EOD prices from CryptoCompare histoday

For each day in daily_calls (Apr 8-13), fetch EOD price at midnight ET.
CryptoCompare `/data/v2/histoday` returns OHLCV for each day.
Use the `close` price at midnight ET as `eod_price`.

In [ ]:
def fetch_eod_price(ticker: str, et_day: str) -> float | None:
    """
    Fetch EOD price for ticker on et_day at midnight ET.
    et_day format: YYYY-MM-DD

    CryptoCompare /data/v2/histoday returns:
    {
        "Data": {
            "TimeFrom": unix_timestamp,
            "TimeTo": unix_timestamp,
            "Data": [ {"close": price, "time": unix, ...}, ... ]
        }
    }
    """
    try:
        # Convert et_day to ET midnight unix timestamp
        et_dt = datetime.strptime(et_day, "%Y-%m-%d").replace(hour=0, minute=0, second=0, tzinfo=ET)
        et_unix = int(et_dt.timestamp())

        api_key = os.environ.get("CRYPTOCOMPARE_API_KEY", "")
        url = "https://min-api.cryptocompare.com/data/v2/histoday"
        params = {
            "fsym": ticker,
            "tsym": "USD",
            "limit": 1,
            "toTs": et_unix,  # Request exactly this timestamp
        }
        if api_key:
            params["api_key"] = api_key

        response = httpx.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()

        if data.get("Response") == "Error":
            print(f"  CryptoCompare error for {ticker} {et_day}: {data.get('Message')}")
            return None

        candles = data.get("Data", {}).get("Data", [])
        if not candles:
            return None

        # Use close price from first candle
        close_price = candles[0].get("close")
        return close_price

    except Exception as e:
        print(f"  Exception fetching {ticker} {et_day}: {e}")
        return None

# Test on one ticker
test_price = fetch_eod_price("VIRTUAL", "2026-04-13")
print(f"Test: VIRTUAL EOD on 2026-04-13 = ${test_price}")

## Step 5: Compute & Store Efficiency Metrics

Once EOD prices are fetched, compute:
- `first_call_efficiency_pct` = (eod - first_price) / first_price * 100
- `last_call_efficiency_pct` = (eod - last_price) / last_price * 100
- `intraday_drift_pct` = (last_price - first_price) / first_price * 100
- `direction_correct` = 1 if (BUY and eod > first) or (SELL and eod < first), else 0
- Clear `dq_eod_missing` flag

In [ ]:
def compute_efficiency(activity_type: str, first_price: float, last_price: float, eod_price: float) -> dict:
    """
    Compute efficiency metrics for a call.
    Returns dict with efficiency fields.
    """
    if eod_price is None or first_price is None:
        return {}

    # Efficiency metrics
    first_eff = (eod_price - first_price) / first_price * 100 if first_price != 0 else 0
    last_eff = (eod_price - last_price) / last_price * 100 if last_price != 0 else 0
    intraday = (last_price - first_price) / first_price * 100 if first_price != 0 else 0

    # Direction correct: BUY call + EOD > first, or SELL call + EOD < first
    direction_correct = None
    if activity_type == "BUY":
        direction_correct = 1 if eod_price > first_price else 0
    elif activity_type == "SELL":
        direction_correct = 1 if eod_price < first_price else 0

    return {
        "first_call_efficiency_pct": round(first_eff, 2),
        "last_call_efficiency_pct": round(last_eff, 2),
        "intraday_drift_pct": round(intraday, 2),
        "direction_correct": direction_correct,
    }

# Test
test = compute_efficiency("BUY", 100, 110, 115)
print(f"Test BUY: first=100, last=110, eod=115 → {test}")
test = compute_efficiency("SELL", 100, 90, 95)
print(f"Test SELL: first=100, last=90, eod=95 → {test}")

## Backfill Loop: Fetch EOD + Compute Efficiency for Apr 8-13

This is the main backfill loop. For each row in daily_calls where eod_price is NULL and et_day < today,
fetch EOD price and compute efficiency fields.

In [ ]:
def backfill_eod_and_efficiency():
    """
    Main backfill loop:
    1. Query rows with eod_price IS NULL and et_day < today
    2. For each row: fetch EOD from CryptoCompare, compute efficiency, UPDATE row
    3. Report stats
    """
    conn = get_conn()
    try:
        today = datetime.now(ET).date().isoformat()

        # Find rows needing EOD backfill
        rows_to_fill = conn.execute("""
            SELECT id, ticker, et_day, activity_type, first_call_price, last_call_price
            FROM daily_calls
            WHERE eod_price IS NULL
            AND et_day < ?
            ORDER BY et_day DESC, ticker
        """, (today,)).fetchall()

        print(f"Found {len(rows_to_fill)} rows to backfill (et_day < {today})")

        updated = 0
        failed = 0
        failed_tickers = set()

        for row in rows_to_fill:
            ticker = row["ticker"]
            et_day = row["et_day"]
            activity_type = row["activity_type"]
            first_price = row["first_call_price"]
            last_price = row["last_call_price"]

            # Skip if prices missing
            if first_price is None or last_price is None:
                print(f"  SKIP {ticker} {et_day} {activity_type}: missing prices")
                failed += 1
                continue

            # Fetch EOD price
            eod_price = fetch_eod_price(ticker, et_day)
            if eod_price is None:
                print(f"  FAIL {ticker} {et_day} {activity_type}: no EOD data")
                failed += 1
                failed_tickers.add(ticker)
                continue

            # Compute efficiency
            eff = compute_efficiency(activity_type, first_price, last_price, eod_price)

            # UPDATE row
            now_utc = datetime.now(timezone.utc).isoformat()
            conn.execute("""
                UPDATE daily_calls
                SET
                    eod_price = ?,
                    eod_fetched_at = ?,
                    first_call_efficiency_pct = ?,
                    last_call_efficiency_pct = ?,
                    intraday_drift_pct = ?,
                    direction_correct = ?,
                    dq_eod_missing = 0
                WHERE id = ?
            """, (
                eod_price,
                now_utc,
                eff.get("first_call_efficiency_pct"),
                eff.get("last_call_efficiency_pct"),
                eff.get("intraday_drift_pct"),
                eff.get("direction_correct"),
                row["id"],
            ))
            updated += 1
            
            if updated % 50 == 0:
                print(f"  Updated {updated}...")

        conn.commit()
        print(f"\n✅ Backfill complete:")
        print(f"   Updated: {updated}")
        print(f"   Failed: {failed}")
        if failed_tickers:
            print(f"   Failed tickers: {', '.join(sorted(failed_tickers))}")

        # Show sample of backfilled rows
        sample = conn.execute("""
            SELECT ticker, et_day, activity_type, eod_price, first_call_efficiency_pct, direction_correct
            FROM daily_calls
            WHERE eod_price IS NOT NULL
            LIMIT 5
        """).fetchall()
        print(f"\nSample backfilled rows:")
        for row in sample:
            print(f"  {row['ticker']:8} {row['et_day']} {row['activity_type']:4} "
                  f"EOD=${row['eod_price']:>8.4f} Eff={row['first_call_efficiency_pct']:>6.2f}% "
                  f"Correct={row['direction_correct']}")

    finally:
        conn.close()

# Run backfill
# backfill_eod_and_efficiency()
print("Ready to call backfill_eod_and_efficiency() to populate EOD + efficiency fields")

## Validation Queries

After backfill, verify that fields are populated and compute analytics.

In [ ]:
def validate_backfill():
    """
    Show backfill statistics and sample analytics.
    """
    conn = get_conn()
    try:
        # Row counts
        total = conn.execute("SELECT COUNT(*) as cnt FROM daily_calls").fetchone()["cnt"]
        with_eod = conn.execute("SELECT COUNT(*) as cnt FROM daily_calls WHERE eod_price IS NOT NULL").fetchone()["cnt"]
        with_mcap = conn.execute("SELECT COUNT(*) as cnt FROM daily_calls WHERE mcap_at_call IS NOT NULL").fetchone()["cnt"]
        with_eff = conn.execute("SELECT COUNT(*) as cnt FROM daily_calls WHERE first_call_efficiency_pct IS NOT NULL").fetchone()["cnt"]

        print(f"Backfill Statistics:")
        print(f"  Total rows: {total}")
        print(f"  With EOD price: {with_eod} ({100*with_eod/total:.1f}%)")
        print(f"  With MCap: {with_mcap} ({100*with_mcap/total:.1f}%)")
        print(f"  With efficiency: {with_eff} ({100*with_eff/total:.1f}%)")

        # Analytics: BUY accuracy
        buy_total = conn.execute("SELECT COUNT(*) as cnt FROM daily_calls WHERE activity_type='BUY' AND direction_correct IS NOT NULL").fetchone()["cnt"]
        buy_correct = conn.execute("SELECT COUNT(*) as cnt FROM daily_calls WHERE activity_type='BUY' AND direction_correct=1").fetchone()["cnt"]
        if buy_total > 0:
            print(f"\n  BUY accuracy: {buy_correct}/{buy_total} ({100*buy_correct/buy_total:.1f}%)")

        # Analytics: SELL accuracy
        sell_total = conn.execute("SELECT COUNT(*) as cnt FROM daily_calls WHERE activity_type='SELL' AND direction_correct IS NOT NULL").fetchone()["cnt"]
        sell_correct = conn.execute("SELECT COUNT(*) as cnt FROM daily_calls WHERE activity_type='SELL' AND direction_correct=1").fetchone()["cnt"]
        if sell_total > 0:
            print(f"  SELL accuracy: {sell_correct}/{sell_total} ({100*sell_correct/sell_total:.1f}%)")

        # Best performing tickers by first_call_efficiency
        top_tickers = conn.execute("""
            SELECT ticker, activity_type, COUNT(*) as count, 
                   ROUND(AVG(first_call_efficiency_pct), 2) as avg_eff,
                   ROUND(AVG(direction_correct)*100, 1) as accuracy
            FROM daily_calls
            WHERE first_call_efficiency_pct IS NOT NULL
            GROUP BY ticker, activity_type
            ORDER BY avg_eff DESC
            LIMIT 10
        """).fetchall()

        print(f"\nTop 10 tickers by efficiency:")
        for row in top_tickers:
            print(f"  {row['ticker']:8} {row['activity_type']:4} n={row['count']:>3} "
                  f"avg_eff={row['avg_eff']:>7.2f}% accuracy={row['accuracy']:>5.1f}%")

    finally:
        conn.close()

# validate_backfill()
print("Ready to call validate_backfill() to show analytics after backfill complete")

## Execution Summary

To complete the backfill:

1. **Call `/metrics/refresh`** from frontend (or curl below)
2. **Run cell:** `update_daily_calls_mcap()` — fills MCap + tier
3. **Run cell:** `backfill_eod_and_efficiency()` — fetches EOD + computes efficiency
4. **Run cell:** `validate_backfill()` — shows analytics

Then analytics endpoints (`/analytics/heatmap/hourly`, `/analytics/windows/4h`) will have real data.

In [ ]:
# Optional: Call /metrics/refresh programmatically
# import requests
# token = os.environ.get("SIGNAL_API_KEY", "changeme_local")
# response = requests.post(
#     "http://localhost:8000/metrics/refresh",
#     params={"all_tickers": "true"},
#     headers={"Authorization": f"Bearer {token}"}
# )
# print(response.json())

print("\n✅ Notebook ready.")
print("Step 3: Uncomment & run update_daily_calls_mcap()")
print("Step 4-5: Uncomment & run backfill_eod_and_efficiency()")
print("Then: Run validate_backfill()")